# Indexing & Retrieval: LangChain (KUHP RAG)

Versi LangChain dari `indexing.ipynb`. Logika `retrieval.py` ada di `src/retrieval_langchain.py` (hanya butuh `langchain-core` + `langchain-ollama`).

Input: `data/processed/chunks.json` (624 pasal, hasil dari `pre-processing.ipynb`).
Output: `data/index/langchain/` berisi `faiss.index`, `embeddings.npy`, `config.json`.

Embedding memakai `BGEM3FlagModel` yang sama dengan versi manual, jadi hasilnya harus identik dengan `data/index/baseline/`. Sel di bagian 4 mengeceknya. Index baseline lama tidak disentuh.

Versi ini pakai `text_for_embedding` polos (tanpa breadcrumb Buku/Bab/Bagian) sebagai baseline.

In [1]:
import sys
sys.path.append("src")

import json
import os

import numpy as np

# `faiss` sengaja TIDAK di-import di sini (lihat catatan di bagian 3).

from retrieval_langchain import (
    EMBEDDING_MODEL,
    expand_query,
    get_embeddings,
    load_documents,
    load_resources,
    rewrite_query,
    search,
)

## 1. Load data & cek batas token

In [2]:
from transformers import AutoTokenizer

docs = load_documents("data/processed/chunks.json")
print(f"Jumlah pasal dimuat: {len(docs)}")

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
panjang_token = [(d.metadata["pasal"], len(tokenizer.encode(d.page_content))) for d in docs]
panjang_token.sort(key=lambda x: x[1], reverse=True)

print("5 pasal dengan token terbanyak:")
for pasal, n_token in panjang_token[:5]:
    print(f"  Pasal {pasal}: {n_token} token")

lebih_dari_batas = [p for p, n in panjang_token if n > 8192]
print(f"Pasal yang melebihi 8192 token: {lebih_dari_batas}")

Jumlah pasal dimuat: 624


5 pasal dengan token terbanyak:
  Pasal 622: 5155 token
  Pasal 609: 719 token
  Pasal 473: 604 token
  Pasal 607: 479 token
  Pasal 85: 463 token
Pasal yang melebihi 8192 token: []


## 2. text_for_embedding (baseline, polos)

`page_content` Document tetap `text` asli (dipakai BM25, reranker, dan LLM). Teks yang di-embed dipisah di `texts_for_embedding`, jadi varian breadcrumb nanti cukup mengganti isi list ini.

In [3]:
texts_for_embedding = [d.page_content for d in docs]  # baseline: identik dengan text
print("text_for_embedding disiapkan (baseline, identik dengan text)")

text_for_embedding disiapkan (baseline, identik dengan text)


## 3. Embedding (BGE-M3) + FAISS

**Urutan penting di macOS:** encode dengan torch dijalankan dulu, `import faiss` baru sesudahnya (di sel pembuatan index). faiss-cpu dan torch masing-masing membawa `libomp`; kalau faiss sudah termuat saat encode berat berjalan, dua runtime OpenMP itu bisa bentrok dan kernel mati (SIGSEGV di `libomp.dylib`). Urutan ini sama dengan `indexing.ipynb` lama. Kalau kernel sudah pernah mati, **Restart** kernel dulu dan jalankan dari sel paling atas.

In [4]:
embeddings = get_embeddings()  # BGEM3Embeddings: BGEM3FlagModel, max_length=8192, batch_size=12, fp32

vecs = np.array(embeddings.embed_documents(texts_for_embedding), dtype="float32")
print(f"Shape embeddings: {vecs.shape}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 52/52 [01:01<00:00,  1.18s/it]


Shape embeddings: (624, 1024)


In [5]:
import faiss  # sengaja di sini, setelah encode selesai

faiss.normalize_L2(vecs)  # defensif — dense_vecs BGE-M3 sebenarnya sudah ternormalisasi

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

print(f"Jumlah vektor di index: {index.ntotal}")

Jumlah vektor di index: 624


In [6]:
# simpan hasil
os.makedirs("data/index/langchain", exist_ok=True)

np.save("data/index/langchain/embeddings.npy", vecs)
faiss.write_index(index, "data/index/langchain/faiss.index")

config = {
    "model": EMBEDDING_MODEL,
    "dim": int(vecs.shape[1]),
    "max_length": 8192,
    "text_for_embedding": "baseline (polos, identik dengan text)",
    "jumlah_pasal": len(docs),
}
with open("data/index/langchain/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Tersimpan ke data/index/langchain/")

Tersimpan ke data/index/langchain/


## 4. Cek kesamaan dengan index baseline (versi manual)

Kalau selisihnya ~0, versi LangChain memakai vektor yang sama dengan versi manual.

In [7]:
path_lama = "data/index/baseline/embeddings.npy"
if os.path.exists(path_lama):
    lama = np.load(path_lama)
    selisih = float(np.abs(lama - vecs).max())
    print(f"Selisih maksimum terhadap baseline: {selisih:.2e}")
    print("SAMA" if selisih < 1e-4 else "BEDA — cek model / text_for_embedding / max_length")
else:
    print("data/index/baseline/embeddings.npy tidak ditemukan, cek dilewati")

Selisih maksimum terhadap baseline: 0.00e+00
SAMA


## 5. BM25

Tidak ada langkah simpan. BM25 dibangun ulang dari `chunks.json` setiap `load_resources()` dipanggil (624 pasal, hitungan milidetik), dengan tokenizer yang sama: lowercase + regex `[a-z0-9]+`, tanpa stemming/stopword removal. `tokenize` ada di `src/retrieval_langchain.py`, jadi tidak ada risiko pickle tidak bisa dibuka.

## 6. Tes retrieval

In [8]:
res = load_resources(index_path="data/index/langchain/faiss.index")

def tampilkan(hasil):
    for d in hasil:
        print(f"  Pasal {d.metadata['pasal']:3d} (skor {d.metadata['score']:.4f}) | {d.metadata['bab']}")

def pasal_list(hasil):
    return [d.metadata["pasal"] for d in hasil]

In [9]:
# tes cepat BM25
print("Tes cepat BM25 'pencurian dengan kekerasan':")
tampilkan(search("pencurian dengan kekerasan", method="bm25", res=res, top_k=5))

Tes cepat BM25 'pencurian dengan kekerasan':
  Pasal 245 (skor 11.4957) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 477 (skor 9.7443) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 479 (skor 9.6543) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 476 (skor 7.9872) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 414 (skor 7.0069) | BAB XV - TINDAK PIDANA KESUSILAAN


In [10]:
# tes 1: routing (harusnya langsung lookup, skip semua retrieval)
print("=== tes routing ===")
tampilkan(search("Pasal 362 itu bagian apa", method="semantic", res=res))

# tes 2: query kasus biasa, tanpa reranker
print("\n=== tanpa reranker (hybrid) ===")
tampilkan(search("nyolong motor tetangga itu pasal apa", method="hybrid", res=res, top_k=5))

# tes 3: query yang sama, DENGAN reranker
print("\n=== dengan reranker (hybrid) ===")
tampilkan(search("nyolong motor tetangga itu pasal apa", method="hybrid", res=res, top_k=5,
                 use_reranker=True))

# tes 4: kasus BM25 yang dulu salah + reranker
print("\n=== kasus BM25 bermasalah dulu, pakai BM25 + reranker ===")
tampilkan(search("pencurian dengan kekerasan", method="bm25", res=res, top_k=5,
                 use_reranker=True))

=== tes routing ===
  Pasal 362 (skor 1.0000) | BAB IX - TINDAK PIDANA TERHADAP KEKUASAAN PEMERINTAHAN

=== tanpa reranker (hybrid) ===
  Pasal 265 (skor 0.0301) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 209 (skor 0.0265) | BAB I - TINDAK PIDANA TERHADAP KEAMANAN NEGARA
  Pasal 312 (skor 0.0241) | BAB VIII - TINDAK PIDANA YANG MEMBAHAYAKAN KEAMANAN UMUM BAGI ORANG, KESEHATAN, DAN BARANG
  Pasal 461 (skor 0.0231) | BAB XXI - TINDAK PIDANA TERHADAP NYAWA DAN JANIN
  Pasal 454 (skor 0.0226) | BAB XIX - TINDAK PIDANA TERHADAP KEMERDEKAAN ORANG

=== dengan reranker (hybrid) ===


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Pasal 265 (skor -3.3509) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 476 (skor -3.4074) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 478 (skor -3.4229) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 461 (skor -3.7255) | BAB XXI - TINDAK PIDANA TERHADAP NYAWA DAN JANIN
  Pasal 486 (skor -4.0074) | BAB XXVI - TINDAK PIDANA PENGGELAPAN

=== kasus BM25 bermasalah dulu, pakai BM25 + reranker ===
  Pasal 479 (skor 3.4772) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 245 (skor 1.5062) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 477 (skor 0.9081) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 262 (skor -0.9804) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 543 (skor -1.3265) | BAB XXXI - TINDAK PIDANA PELAYARAN


## 7. Bandingkan dengan output versi manual

Urutan pasal di bawah disalin dari output `retrieval.py` di notebook lama. Skor RRF dan skor reranker juga harus sama (misal hybrid Pasal 265 = 0.0301, dengan reranker = -3.3509).

In [11]:
kasus = [
    ("bm25 'pencurian dengan kekerasan'",
     lambda: search("pencurian dengan kekerasan", "bm25", res, top_k=5),
     [245, 477, 479, 476, 414]),
    ("routing 'Pasal 362 itu bagian apa'",
     lambda: search("Pasal 362 itu bagian apa", "semantic", res),
     [362]),
    ("hybrid 'nyolong motor tetangga itu pasal apa'",
     lambda: search("nyolong motor tetangga itu pasal apa", "hybrid", res, top_k=5),
     [265, 209, 312, 461, 454]),
    ("hybrid + reranker 'nyolong motor tetangga itu pasal apa'",
     lambda: search("nyolong motor tetangga itu pasal apa", "hybrid", res, top_k=5, use_reranker=True),
     [265, 476, 478, 461, 486]),
    ("bm25 + reranker 'pencurian dengan kekerasan'",
     lambda: search("pencurian dengan kekerasan", "bm25", res, top_k=5, use_reranker=True),
     [479, 245, 477, 262, 543]),
]

for nama, jalankan, harapan in kasus:
    hasil = pasal_list(jalankan())
    status = "SAMA" if hasil == harapan else "BEDA"
    print(f"[{status}] {nama}")
    if hasil != harapan:
        print(f"    manual   : {harapan}")
        print(f"    langchain: {hasil}")

[SAMA] bm25 'pencurian dengan kekerasan'
[SAMA] routing 'Pasal 362 itu bagian apa'
[SAMA] hybrid 'nyolong motor tetangga itu pasal apa'
[SAMA] hybrid + reranker 'nyolong motor tetangga itu pasal apa'
[SAMA] bm25 + reranker 'pencurian dengan kekerasan'


In [12]:
# tes rewriting & expansion (butuh `ollama serve` jalan)
q = "nyolong motor tetangga itu pasal apa"
print("Rewrite:", rewrite_query(q))
print("Expand :", expand_query(q))

Rewrite: Apa dasar hukum yang mengatur mengenai tindak pidana pencurian sepeda motor milik tetangga?
Expand : nyolong motor tetangga itu pasal apa Pencurian, Pencurian kendaraan bermotor
